# Track B Complete Pipeline (0 to 100)

This notebook covers the **full Track B workflow** for Short-Term Anticipation:

### Pipeline Overview:
1. **Environment Setup** - Check CUDA, paths, dependencies
2. **Data Validation** - Verify frames, manifests, annotations
3. **Backbone Selection** - ResNet18 vs VideoMAE
4. **Tokenization** - Extract visual features from frames
5. **Dataset Creation** - Load manifests and build dataset
6. **Model Architecture** - Projector → Fusion → Head
7. **Training Loop** - Full training with validation
8. **Evaluation** - Metrics, predictions, overlays
9. **Results Analysis** - Plots and performance summary

### Track B Components:
- `trackB_tokenizer.py` - Frame/video tokenization (ResNet18 or VideoMAE)
- `trackB_dataset.py` - Dataset + collate with TTC normalization
- `trackB_fusion.py` - Frame-Guided Temporal Pooling + Cross-Attention
- `trackB_head.py` - Classification + TTC regression heads
- `trackB_train_loader.py` - Training with early stopping
- `trackB_eval.py` - Evaluation with hotspot priors + CLIP reranking

---
## Part 1: Environment Setup

In [29]:
# 1.1 Check System & GPU
import torch
import sys
from pathlib import Path

print("=" * 50)
print("SYSTEM CHECK")
print("=" * 50)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = "cuda"
else:
    print("⚠️  Running on CPU - training will be slow!")
    DEVICE = "cpu"

print(f"\nUsing device: {DEVICE}")

SYSTEM CHECK
Python: 3.13.3
PyTorch: 2.9.0+cpu
CUDA available: False
⚠️  Running on CPU - training will be slow!

Using device: cpu


In [30]:
# 1.2 Setup Paths
import os
import sys
from pathlib import Path

# Find local_extraction directory
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    LOCAL_EXT = notebook_dir.parent / 'local_extraction'
else:
    LOCAL_EXT = Path(r'D:\Thesis\Ego4d-LiteSTA\local_extraction')

REPO_ROOT = LOCAL_EXT.parent

# Add to Python path
sys.path.insert(0, str(LOCAL_EXT))
sys.path.insert(0, str(REPO_ROOT))

os.chdir(LOCAL_EXT)
print(f"Working directory: {Path.cwd()}")

# Define all paths
PATHS = {
    'frames': LOCAL_EXT / 'v2' / 'extracted_frames',
    'manifests': LOCAL_EXT / 'v2' / 'manifests',
    'trackA_runs': LOCAL_EXT / 'runs' / 'Track_A',
    'trackB_runs': LOCAL_EXT / 'runs' / 'Track_B',
    'configs': LOCAL_EXT / 'configs',
    'annotations': LOCAL_EXT / 'v2' / 'org_annotations',
}

print("\n" + "=" * 50)
print("PATH VALIDATION")
print("=" * 50)
for name, path in PATHS.items():
    status = "✓" if path.exists() else "✗"
    print(f"{status} {name}: {path}")

# Count available data
if PATHS['frames'].exists():
    uid_count = len(list(PATHS['frames'].iterdir()))
    print(f"\n📁 Found {uid_count} UIDs with extracted frames")

Working directory: D:\Thesis\Ego4d-LiteSTA\local_extraction

PATH VALIDATION
✓ frames: D:\Thesis\Ego4d-LiteSTA\local_extraction\v2\extracted_frames
✓ manifests: D:\Thesis\Ego4d-LiteSTA\local_extraction\v2\manifests
✓ trackA_runs: D:\Thesis\Ego4d-LiteSTA\local_extraction\runs\Track_A
✓ trackB_runs: D:\Thesis\Ego4d-LiteSTA\local_extraction\runs\Track_B
✓ configs: D:\Thesis\Ego4d-LiteSTA\local_extraction\configs
✓ annotations: D:\Thesis\Ego4d-LiteSTA\local_extraction\v2\org_annotations

📁 Found 2324 UIDs with extracted frames


---
## Part 2: Configuration

In [31]:
# 2.1 Select Backbone
# ========================================
# CHOOSE YOUR BACKBONE HERE
# ========================================
BACKBONE = "resnet18"  # Options: "resnet18" or "videomae"
# ========================================

# Backbone specifications
BACKBONE_SPECS = {
    "resnet18": {
        "dim": 512,
        "grid": (7, 7),
        "frames": 8,
        "description": "Fast, lightweight CNN backbone",
    },
    "videomae": {
        "dim": 768,
        "grid": (14, 14),
        "frames": 16,
        "description": "Video Masked Autoencoder (Transformer)",
    }
}

spec = BACKBONE_SPECS[BACKBONE]
print("=" * 50)
print(f"BACKBONE: {BACKBONE.upper()}")
print("=" * 50)
print(f"Description: {spec['description']}")
print(f"Token dimension: {spec['dim']}")
print(f"Grid size: {spec['grid']} = {spec['grid'][0] * spec['grid'][1]} tokens")
print(f"Frames per sample: {spec['frames']}")

BACKBONE: RESNET18
Description: Fast, lightweight CNN backbone
Token dimension: 512
Grid size: (7, 7) = 49 tokens
Frames per sample: 8


In [ ]:
# 2.2 Training Configuration
from dataclasses import dataclass
from typing import Optional

# ========================================
# DEMO MODE TOGGLE
# ========================================
DEMO_MODE = True  # Set to False for full training
# ========================================

@dataclass
class Config:
    """Training configuration - modify these as needed"""
    
    # === BACKBONE ===
    backbone: str = BACKBONE
    token_dim: int = 256  # Projector output dimension
    
    # === TRAINING ===
    # Demo mode: fast test with minimal epochs/batches
    # Full mode: proper training settings
    epochs: int = 2 if DEMO_MODE else 20
    batch_size: int = 2 if DEMO_MODE else 4
    lr: float = 1e-3
    min_lr: float = 1e-5
    warmup_epochs: float = 0.5 if DEMO_MODE else 1.0
    
    # === MIXED PRECISION (AMP) ===
    use_amp: bool = False  # Enable Automatic Mixed Precision for faster training
    # Note: AMP works best on GPUs with Tensor Cores (V100, A100, RTX 20xx+)
    
    # === DATA ===
    candidate_limit: int = 8 if DEMO_MODE else 16  # Max candidates per frame
    normalize_ttc: bool = True
    label_smoothing: float = 0.05
    
    # === MODEL ===
    fusion_layers: int = 1 if DEMO_MODE else 2
    fusion_heads: int = 4 if DEMO_MODE else 8
    head_hidden: int = 128 if DEMO_MODE else 256
    num_classes: int = 2  # Binary: next-active or not
    
    # === MULTI-TASK LOSS WEIGHTS ===
    # From trackB_train_loader.py - weighted sum of losses
    use_multi_task: bool = True   # Enable noun/verb/TTC heads
    loss_w_next: float = 1.5      # Next-active classification weight
    loss_w_noun: float = 0.25     # Noun classification weight
    loss_w_verb: float = 0.25     # Verb classification weight
    loss_w_ttc: float = 0.1       # TTC regression weight (scaled down)
    
    # === VALIDATION ===
    eval_every: int = 1
    early_stopping_patience: int = 2 if DEMO_MODE else 5
    monitor_metric: str = "mAP"  # mAP, accuracy, or ttc_mae
    
    # === DEVICE ===
    device: str = DEVICE

cfg = Config()

print("=" * 50)
if DEMO_MODE:
    print("🧪 DEMO MODE - Fast test run")
else:
    print("🚀 FULL TRAINING MODE")
print("=" * 50)
print(f"Backbone: {cfg.backbone} ({spec['dim']}d → {cfg.token_dim}d)")
print(f"Epochs: {cfg.epochs}")
print(f"Batch size: {cfg.batch_size}")
print(f"Learning rate: {cfg.lr} → {cfg.min_lr}")
print(f"Fusion: {cfg.fusion_layers} layers, {cfg.fusion_heads} heads")
print(f"Multi-task: {cfg.use_multi_task}")
print(f"Mixed Precision (AMP): {cfg.use_amp}")
print(f"Device: {cfg.device}")

if cfg.use_multi_task:
    print(f"\n📊 Multi-Task Loss Weights:")
    print(f"   Next-active: {cfg.loss_w_next}")
    print(f"   Noun:        {cfg.loss_w_noun}")
    print(f"   Verb:        {cfg.loss_w_verb}")
    print(f"   TTC:         {cfg.loss_w_ttc}")

if DEMO_MODE:
    print("\n⚠️  To run full training, set DEMO_MODE = False")

🧪 DEMO MODE - Fast test run
Backbone: resnet18 (512d → 256d)
Epochs: 2
Batch size: 2
Learning rate: 0.001 → 1e-05
Fusion: 1 layers, 4 heads
Multi-task: True
Device: cpu

⚠️  To run full training, set DEMO_MODE = False


---
## Part 3: Data Validation

In [33]:
pip install torchvision

Note: you may need to restart the kernel to use updated packages.


In [34]:
# 3.1 Import Track B Modules
from trackB.trackB_tokenizer import (
    TokenizerConfig, build_backbone, build_transform, get_tokens,
    roi_pool_tokens_mean, list_uid_frames, sample_window_ending_at
)
from trackB.trackB_dataset import (
    TrackBDataset, trackB_collate, latest_stageB_run, resolve_stageB_manifest
)
from trackB.trackB_fusion import FusionConfig, TrackBFusion
from trackB.trackB_head import HeadConfig, TrackBHead
from trackB.trackB_metrics import (
    binary_accuracy, multiclass_accuracy, binary_average_precision,
    per_class_ap, ttc_mae, denorm_ttc
)

print("✓ All Track B modules imported successfully")

✓ All Track B modules imported successfully


In [35]:
# 3.2 Find Training Manifests
import json

# Use specific Stage B run (not auto-discovery)
STAGEB_RUN_NAME = "trackA_stageB_20251117_184342"
stageB_run = PATHS['trackA_runs'] / STAGEB_RUN_NAME

print("=" * 50)
print("MANIFEST DISCOVERY")
print("=" * 50)

if stageB_run.exists():
    print(f"✓ Using Stage B run: {stageB_run.name}")
    
    train_manifest = stageB_run / 'head_train.jsonl'
    val_manifest = stageB_run / 'head_val.jsonl'
    
    print(f"\n{'✓' if train_manifest.exists() else '✗'} Train: {train_manifest.name}")
    print(f"{'✓' if val_manifest.exists() else '✗'} Val: {val_manifest.name}")
    
    # Count records
    if train_manifest.exists():
        with open(train_manifest) as f:
            train_records = [json.loads(l) for l in f]
        print(f"\n📊 Train samples: {len(train_records)}")
        
        # Show sample record structure
        sample = train_records[0]
        print(f"\nSample record keys: {list(sample.keys())}")
        if 'candidates' in sample:
            print(f"Candidates per frame: {len(sample['candidates'])}")
    
    if val_manifest.exists():
        with open(val_manifest) as f:
            val_count = sum(1 for _ in f)
        print(f"📊 Val samples: {val_count}")
else:
    print(f"✗ Stage B run not found: {stageB_run}")
    print("  Check that the run directory exists")

MANIFEST DISCOVERY
✓ Using Stage B run: trackA_stageB_20251117_184342

✓ Train: head_train.jsonl
✓ Val: head_val.jsonl

📊 Train samples: 3352

Sample record keys: ['image_path', 'candidate_box', 'candidate_conf', 'candidate_cls', 'verb_id', 'noun_id', 'ttc', 'is_positive', 'iou']
📊 Val samples: 1777


In [36]:
# 3.3 Validate Frame Coverage
import random

print("=" * 50)
print("FRAME COVERAGE CHECK")
print("=" * 50)

if train_manifest.exists():
    manifest_uids = set(r.get('uid', r.get('video_uid')) for r in train_records)
    frame_uids = set(p.name for p in PATHS['frames'].iterdir())
    
    covered = manifest_uids & frame_uids
    missing = manifest_uids - frame_uids
    
    print(f"Manifest UIDs: {len(manifest_uids)}")
    print(f"Frame UIDs: {len(frame_uids)}")
    print(f"Covered: {len(covered)} ({100*len(covered)/len(manifest_uids):.1f}%)")
    
    if missing:
        print(f"⚠️  Missing frames for {len(missing)} UIDs")
    else:
        print("✓ All manifest UIDs have frames!")

FRAME COVERAGE CHECK
Manifest UIDs: 1
Frame UIDs: 2324
Covered: 0 (0.0%)
⚠️  Missing frames for 1 UIDs
Manifest UIDs: 1
Frame UIDs: 2324
Covered: 0 (0.0%)
⚠️  Missing frames for 1 UIDs


---
## Part 4: Tokenization Pipeline

In [37]:
# 4.1 Build Tokenizer
import time

tok_cfg = TokenizerConfig()
tok_cfg.video_backbone = cfg.backbone
tok_cfg.num_frames = spec['frames']
tok_cfg.device = cfg.device

print("=" * 50)
print("TOKENIZER CONFIGURATION")
print("=" * 50)
print(f"Backbone: {tok_cfg.video_backbone}")
print(f"Frames: {tok_cfg.num_frames}")
print(f"Device: {tok_cfg.device}")

print("\nBuilding backbone...")
start = time.time()
backbone = build_backbone(tok_cfg.device)  # Uses ResNet18 by default
transform = build_transform(tok_cfg)
print(f"✓ Backbone ready in {time.time() - start:.1f}s")

TOKENIZER CONFIGURATION
Backbone: resnet18
Frames: 8
Device: cpu

Building backbone...
✓ Backbone ready in 0.6s
✓ Backbone ready in 0.6s


In [38]:
# 4.2 Test Single Frame Tokenization
import time

# Get a sample frame
sample_uid_dir = next(PATHS['frames'].iterdir())
sample_frames = sorted([f for f in sample_uid_dir.iterdir() if f.suffix == '.jpg'])
sample_frame = sample_frames[-1]

print("=" * 50)
print("TOKENIZATION TEST")
print("=" * 50)
print(f"Test frame: {sample_frame.parent.name}/{sample_frame.name}")

# Tokenize
start = time.time()
img_tokens, vid_tokens, hw = get_tokens(
    sample_frame, PATHS['frames'], tok_cfg, backbone, transform
)
elapsed = time.time() - start

print(f"\n✓ Tokenization complete in {elapsed:.2f}s")
print(f"\nImage tokens: {tuple(img_tokens.shape)}")
print(f"  → {img_tokens.shape[0]} spatial tokens, {img_tokens.shape[1]}d features")
print(f"\nVideo tokens: {tuple(vid_tokens.shape)}")
print(f"  → {vid_tokens.shape[0]} frames × {vid_tokens.shape[1]} tokens × {vid_tokens.shape[2]}d")
print(f"\nGrid size: {hw} = {hw[0]}×{hw[1]} tokens")

TOKENIZATION TEST
Test frame: 002e11bc-deef-45f7-9af8-59421a606d69/0008939.jpg

✓ Tokenization complete in 1.96s

Image tokens: (49, 512)
  → 49 spatial tokens, 512d features

Video tokens: (8, 49, 512)
  → 8 frames × 49 tokens × 512d

Grid size: (7, 7) = 7×7 tokens

✓ Tokenization complete in 1.96s

Image tokens: (49, 512)
  → 49 spatial tokens, 512d features

Video tokens: (8, 49, 512)
  → 8 frames × 49 tokens × 512d

Grid size: (7, 7) = 7×7 tokens


---
## Part 5: Dataset Creation

In [39]:
# 5.1 Create Training Dataset
print("=" * 50)
print("DATASET CREATION")
print("=" * 50)

train_ds = TrackBDataset(
    frames_root=PATHS['frames'],
    manifests_root=stageB_run or PATHS['trackA_runs'],
    manifest_path=train_manifest,
    tokenizer_cfg=tok_cfg,
    candidate_limit=cfg.candidate_limit,
    normalize_ttc=cfg.normalize_ttc,
)

print(f"✓ Training dataset: {len(train_ds)} samples")
print(f"  Candidate limit: {cfg.candidate_limit}")
print(f"  TTC normalization: {cfg.normalize_ttc}")

# TTC statistics
if hasattr(train_ds, 'ttc_stats') and train_ds.ttc_stats:
    stats = train_ds.ttc_stats
    print(f"\nTTC Statistics:")
    print(f"  Mean: {stats.get('mean', 0):.2f}s")
    print(f"  Std: {stats.get('std', 0):.2f}s")
    print(f"  Range: [{stats.get('min', 0):.2f}, {stats.get('max', 0):.2f}]s")

DATASET CREATION


[TrackBDataset] parse records: 100%|██████████| 1490/1490 [00:20<00:00, 72.48it/s] 

✓ Training dataset: 1490 samples
  Candidate limit: 16
  TTC normalization: True


In [40]:
# 5.2 Create Validation Dataset
if val_manifest.exists():
    val_ds = TrackBDataset(
        frames_root=PATHS['frames'],
        manifests_root=stageB_run or PATHS['trackA_runs'],
        manifest_path=val_manifest,
        tokenizer_cfg=tok_cfg,
        candidate_limit=cfg.candidate_limit,
        normalize_ttc=cfg.normalize_ttc,
    )
    print(f"✓ Validation dataset: {len(val_ds)} samples")
else:
    val_ds = None
    print("⚠️  No validation manifest found")

[TrackBDataset] parse records: 100%|██████████| 755/755 [00:05<00:00, 139.22it/s]

✓ Validation dataset: 755 samples


In [41]:
# 5.3 Test Dataset Loading
import time
from torch.utils.data import DataLoader

print("=" * 50)
print("DATALOADER TEST")
print("=" * 50)

# Create dataloaders
train_loader = DataLoader(
    train_ds, 
    batch_size=cfg.batch_size, 
    shuffle=True, 
    num_workers=0,  # Must be 0 for tokenization in __getitem__
    collate_fn=trackB_collate
)

if val_ds:
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=trackB_collate
    )

# Test one batch
print(f"Testing batch load (batch_size={cfg.batch_size})...")
start = time.time()
batch = next(iter(train_loader))
elapsed = time.time() - start

if batch.get('valid'):
    print(f"✓ Batch loaded in {elapsed:.2f}s")
    print(f"  Samples in batch: {len(batch['samples'])}")
    
    # Inspect first sample
    sample = batch['samples'][0]
    print(f"\nSample structure:")
    print(f"  UID: {sample['uid']}")
    print(f"  img_tokens: {tuple(sample['img_tokens'].shape)}")
    print(f"  vid_tokens: {tuple(sample['vid_tokens'].shape)}")
    print(f"  Candidates: {len(sample['bboxes'])}")
    print(f"  Positive: {sample['is_positive'].sum().item()}/{len(sample['is_positive'])}")
else:
    print("✗ Invalid batch")

DATALOADER TEST
Testing batch load (batch_size=4)...
✓ Batch loaded in 5.20s
  Samples in batch: 4

Sample structure:
  UID: 7531c3d2-099a-430e-9403-114c8824e5f3
  img_tokens: (49, 512)
  vid_tokens: (8, 49, 512)
  Candidates: 4
  Positive: 1/4
✓ Batch loaded in 5.20s
  Samples in batch: 4

Sample structure:
  UID: 7531c3d2-099a-430e-9403-114c8824e5f3
  img_tokens: (49, 512)
  vid_tokens: (8, 49, 512)
  Candidates: 4
  Positive: 1/4


---
## Part 6: Model Architecture

In [42]:
# 6.1 Build Model Components
import torch.nn as nn

print("=" * 50)
print("MODEL ARCHITECTURE")
print("=" * 50)

IN_DIM = spec['dim']  # 512 for ResNet18, 768 for VideoMAE

# 1. Projector: Backbone dim → Token dim
projector = nn.Linear(IN_DIM, cfg.token_dim)

# 2. Fusion: Frame-Guided Temporal Pooling + Cross-Attention
fusion_cfg = FusionConfig(
    dim=cfg.token_dim,
    layers=cfg.fusion_layers,
    heads=cfg.fusion_heads,
)
fusion = TrackBFusion(fusion_cfg)

# 3. Head: Classification + TTC regression
head_cfg = HeadConfig(
    dim=cfg.token_dim,
    num_classes=cfg.num_classes,
    hidden=cfg.head_hidden,
)
head = TrackBHead(head_cfg)

# Move to device
projector.to(cfg.device)
fusion.to(cfg.device)
head.to(cfg.device)

# Count parameters
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n1. Projector: {IN_DIM}d → {cfg.token_dim}d")
print(f"   Parameters: {count_params(projector):,}")

print(f"\n2. Fusion: {cfg.fusion_layers} layers, {cfg.fusion_heads} heads")
print(f"   Parameters: {count_params(fusion):,}")

print(f"\n3. Head: {cfg.num_classes} classes + TTC")
print(f"   Parameters: {count_params(head):,}")

total = count_params(projector) + count_params(fusion) + count_params(head)
print(f"\n📊 Total trainable parameters: {total:,}")

MODEL ARCHITECTURE

1. Projector: 512d → 256d
   Parameters: 131,328

2. Fusion: 2 layers, 8 heads
   Parameters: 3,356,928

3. Head: 2 classes + TTC
   Parameters: 133,123

📊 Total trainable parameters: 3,621,379


In [43]:
# 6.2 Test Forward Pass
print("=" * 50)
print("FORWARD PASS TEST")
print("=" * 50)

sample = batch['samples'][0]
img_tok = sample['img_tokens'].to(cfg.device)
vid_tok = sample['vid_tokens'].to(cfg.device)

# 1. Project tokens
img_proj = projector(img_tok.unsqueeze(0))  # (1, N, C)
vid_proj = projector(vid_tok)  # (T, N, C)
print(f"1. Projected: img={tuple(img_proj.shape)}, vid={tuple(vid_proj.shape)}")

# 2. Fuse
fused_img, fused_vid = fusion(img_proj, vid_proj.unsqueeze(0))
print(f"2. Fused: img={tuple(fused_img.shape)}, vid={tuple(fused_vid.shape)}")

# 3. ROI pool for one candidate
if sample['bboxes']:
    box = sample['bboxes'][0]
    hw = sample['hw']
    img_wh = sample['image_size']
    
    roi_feat = roi_pool_tokens_mean(hw, fused_img.squeeze(0), box, img_wh)
    print(f"3. ROI pooled: {tuple(roi_feat.shape)}")
    
    # 4. Head forward
    roi_batch = roi_feat.unsqueeze(0).unsqueeze(0)  # (1, 1, C)
    outputs = head(roi_batch, roi_batch)
    
    print(f"4. Head outputs:")
    print(f"   cls_logits: {tuple(outputs['cls_logits'].shape)}")
    print(f"   ttc: {tuple(outputs['ttc'].shape)}")
    
print("\n✓ Forward pass successful!")

FORWARD PASS TEST
1. Projected: img=(1, 49, 256), vid=(8, 49, 256)
2. Fused: img=(1, 49, 256), vid=(1, 49, 256)
3. ROI pooled: (256,)
4. Head outputs:
   cls_logits: (1, 1, 2)
   ttc: (1, 1, 1)

✓ Forward pass successful!
2. Fused: img=(1, 49, 256), vid=(1, 49, 256)
3. ROI pooled: (256,)
4. Head outputs:
   cls_logits: (1, 1, 2)
   ttc: (1, 1, 1)

✓ Forward pass successful!


---
## Part 7: Training Loop

In [ ]:
# 7.1 Training Setup
import torch.optim as optim
from tqdm import tqdm
import math
from datetime import datetime
import json

print("=" * 50)
print("TRAINING SETUP")
print("=" * 50)

# ═══════════════════════════════════════════════════════════════
# Top-5 Accuracy Function (for multiclass evaluation)
# ═══════════════════════════════════════════════════════════════
def top_k_accuracy(logits: torch.Tensor, labels: torch.Tensor, k: int = 5) -> float:
    """
    Compute top-k accuracy for multiclass classification.
    
    Args:
        logits: (N, num_classes) raw logits or probabilities
        labels: (N,) ground truth class indices
        k: number of top predictions to consider
    
    Returns:
        Fraction of samples where the true label is in top-k predictions
    """
    if logits.numel() == 0:
        return 0.0
    
    # Handle binary case
    if logits.shape[1] == 2:
        # For binary, top-5 = top-2 = basically top-1 or top-2
        k = min(k, 2)
    
    # Get top-k predicted classes
    _, topk_preds = torch.topk(logits, k=min(k, logits.shape[1]), dim=1)
    
    # Check if true label is in top-k
    labels_expanded = labels.view(-1, 1).expand_as(topk_preds)
    correct = (topk_preds == labels_expanded).any(dim=1).float()
    
    return correct.mean().item()


def top5_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    """Convenience wrapper for top-5 accuracy."""
    return top_k_accuracy(logits, labels, k=5)


def compute_all_metrics(logits: torch.Tensor, labels: torch.Tensor) -> dict:
    """
    Compute comprehensive metrics for evaluation.
    
    Returns dict with:
        - accuracy: top-1 accuracy
        - top5_accuracy: top-5 accuracy
        - mAP: binary average precision
    """
    metrics = {
        'accuracy': binary_accuracy(logits, labels),
        'mAP': binary_average_precision(logits, labels),
        'top1_accuracy': multiclass_accuracy(logits, labels) if logits.shape[1] > 2 else binary_accuracy(logits, labels),
        'top5_accuracy': top5_accuracy(logits, labels),
    }
    return metrics


# ═══════════════════════════════════════════════════════════════
# Optimizer Setup
# ═══════════════════════════════════════════════════════════════
all_params = (
    list(projector.parameters()) + 
    list(fusion.parameters()) + 
    list(head.parameters())
)
optimizer = optim.AdamW(all_params, lr=cfg.lr, weight_decay=0.01)

# Loss functions
ce_loss = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
mse_loss = nn.MSELoss()

# Learning rate scheduler with warmup
def get_lr(epoch, total_epochs, warmup_epochs, base_lr, min_lr):
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))

# Output directory (includes backbone name for comparison)
run_name = f"run_{cfg.backbone}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_dir = PATHS['trackB_runs'] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
ckpt_dir = run_dir / "checkpoints"
ckpt_dir.mkdir(exist_ok=True)

# Results file for backbone comparison
RESULTS_FILE = PATHS['trackB_runs'] / "backbone_comparison.json"

print(f"Optimizer: AdamW (lr={cfg.lr})")
print(f"Scheduler: Cosine with {cfg.warmup_epochs} warmup epochs")
print(f"Output: {run_dir}")
print(f"Results comparison file: {RESULTS_FILE}")

TRAINING SETUP
Optimizer: AdamW (lr=0.001)
Scheduler: Cosine with 1.0 warmup epochs
Output: D:\Thesis\Ego4d-LiteSTA\local_extraction\runs\Track_B\run_20251205_032744


In [ ]:
# 7.2 Training Function (with Multi-Task Losses + AMP)
"""
Multi-task training with weighted loss combination:
  Total Loss = w_next * L_next + w_noun * L_noun + w_verb * L_verb + w_ttc * L_ttc

AMP (Automatic Mixed Precision):
  - Uses float16 for forward pass (faster on modern GPUs)
  - Uses float32 for backward pass (numerical stability)
  - Enable with cfg.use_amp = True
"""

# Initialize AMP scaler if enabled
if cfg.use_amp and cfg.device == 'cuda':
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    print("✓ AMP (Mixed Precision) enabled with GradScaler")
else:
    scaler = None
    if cfg.use_amp:
        print("⚠️  AMP requested but not using CUDA - disabled")


def train_one_epoch(epoch):
    """
    Train for one epoch with multi-task losses and optional AMP.
    
    Returns:
        dict with loss, accuracy, mAP, lr, and per-task losses
    """
    projector.train()
    fusion.train()
    head.train()
    
    # Accumulators
    total_loss = 0.0
    total_loss_next = 0.0
    total_loss_noun = 0.0
    total_loss_verb = 0.0
    total_loss_ttc = 0.0
    total_samples = 0
    all_preds = []
    all_labels = []
    
    # Learning rate schedule
    lr = get_lr(epoch, cfg.epochs, cfg.warmup_epochs, cfg.lr, cfg.min_lr)
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs}")
    
    for batch in pbar:
        if not batch.get('valid'):
            continue
        
        optimizer.zero_grad()
        
        # Use AMP context if enabled
        amp_context = autocast() if scaler is not None else torch.enable_grad()
        
        with amp_context:
            batch_loss = torch.tensor(0.0, device=cfg.device)
            batch_loss_next = torch.tensor(0.0, device=cfg.device)
            batch_loss_noun = torch.tensor(0.0, device=cfg.device)
            batch_loss_verb = torch.tensor(0.0, device=cfg.device)
            batch_loss_ttc = torch.tensor(0.0, device=cfg.device)
            count = 0
            
            for sample in batch['samples']:
                if not sample.get('valid'):
                    continue
                
                img_tok = sample['img_tokens'].to(cfg.device)
                vid_tok = sample['vid_tokens'].to(cfg.device)
                
                img_proj = projector(img_tok.unsqueeze(0))
                vid_proj = projector(vid_tok)
                fused_img, fused_vid = fusion(img_proj, vid_proj.unsqueeze(0))
                
                for i, box in enumerate(sample['bboxes']):
                    roi_feat = roi_pool_tokens_mean(
                        sample['hw'], fused_img.squeeze(0), box, sample['image_size']
                    )
                    roi_batch = roi_feat.unsqueeze(0).unsqueeze(0)
                    outputs = head(roi_batch, roi_batch)
                    
                    # ═══════════════════════════════════════════════════
                    # NEXT-ACTIVE LOSS (primary task)
                    # ═══════════════════════════════════════════════════
                    label = sample['is_positive'][i:i+1].to(cfg.device)
                    loss_next = ce_loss(outputs['cls_logits'].squeeze(1), label)
                    batch_loss_next += loss_next
                    
                    # Collect predictions for metrics
                    pred = torch.softmax(outputs['cls_logits'].squeeze(1), dim=-1)[:, 1]
                    all_preds.append(pred.detach().cpu())
                    all_labels.append(label.cpu())
                    
                    # ═══════════════════════════════════════════════════
                    # MULTI-TASK LOSSES (if enabled and available)
                    # ═══════════════════════════════════════════════════
                    if cfg.use_multi_task:
                        # Noun loss
                        if 'noun_logits' in outputs and 'gt_noun_id' in sample:
                            gt_noun = sample['gt_noun_id']
                            if gt_noun is not None and i < len(gt_noun):
                                noun_target = gt_noun[i:i+1].to(cfg.device).long()
                                if noun_target.item() >= 0:  # Valid noun
                                    loss_noun = ce_loss(outputs['noun_logits'].squeeze(1), noun_target)
                                    batch_loss_noun += loss_noun
                        
                        # Verb loss
                        if 'verb_logits' in outputs and 'gt_verb_id' in sample:
                            gt_verb = sample['gt_verb_id']
                            if gt_verb is not None and i < len(gt_verb):
                                verb_target = gt_verb[i:i+1].to(cfg.device).long()
                                if verb_target.item() >= 0:  # Valid verb
                                    loss_verb = ce_loss(outputs['verb_logits'].squeeze(1), verb_target)
                                    batch_loss_verb += loss_verb
                        
                        # TTC loss (regression)
                        if 'ttc' in outputs and 'ttc' in sample:
                            gt_ttc = sample['ttc']
                            if gt_ttc is not None and i < len(gt_ttc):
                                ttc_target = gt_ttc[i:i+1].to(cfg.device).float()
                                if not torch.isnan(ttc_target):
                                    ttc_pred = outputs['ttc'].squeeze()
                                    loss_ttc = mse_loss(ttc_pred, ttc_target)
                                    batch_loss_ttc += loss_ttc
                    
                    count += 1
            
            if count > 0:
                # Weighted sum of losses
                batch_loss = (
                    cfg.loss_w_next * batch_loss_next +
                    cfg.loss_w_noun * batch_loss_noun +
                    cfg.loss_w_verb * batch_loss_verb +
                    cfg.loss_w_ttc * batch_loss_ttc
                ) / count
        
        # Backward pass with AMP scaling if enabled
        if count > 0:
            if scaler is not None:
                scaler.scale(batch_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                batch_loss.backward()
                torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
                optimizer.step()
            
            total_loss += batch_loss.item() * count
            total_loss_next += batch_loss_next.item()
            total_loss_noun += batch_loss_noun.item()
            total_loss_verb += batch_loss_verb.item()
            total_loss_ttc += batch_loss_ttc.item()
            total_samples += count
            
            pbar.set_postfix({
                'loss': f"{batch_loss.item():.4f}",
                'next': f"{batch_loss_next.item()/count:.3f}",
                'lr': f"{lr:.2e}"
            })
    
    # Compute epoch metrics
    if all_preds:
        preds = torch.cat(all_preds)
        labels = torch.cat(all_labels)
        acc = binary_accuracy(preds, labels)
        ap = binary_average_precision(preds, labels)
    else:
        acc = ap = 0.0
    
    return {
        'loss': total_loss / max(total_samples, 1),
        'loss_next': total_loss_next / max(total_samples, 1),
        'loss_noun': total_loss_noun / max(total_samples, 1),
        'loss_verb': total_loss_verb / max(total_samples, 1),
        'loss_ttc': total_loss_ttc / max(total_samples, 1),
        'accuracy': acc,
        'mAP': ap,
        'lr': lr,
    }

In [ ]:
# 7.3 Validation Function (with Top-5 Metrics)
@torch.no_grad()
def validate():
    """
    Run validation, returns comprehensive metrics dict including top-5 accuracy.
    
    Returns:
        dict with: accuracy, mAP, top5_accuracy, ttc_mae
    """
    if val_ds is None:
        return None
    
    projector.eval()
    fusion.eval()
    head.eval()
    
    all_logits = []  # Store raw logits for top-k computation
    all_preds = []
    all_labels = []
    all_ttc_preds = []
    all_ttc_labels = []
    
    for batch in tqdm(val_loader, desc="Validation"):
        if not batch.get('valid'):
            continue
        
        for sample in batch['samples']:
            if not sample.get('valid'):
                continue
            
            img_tok = sample['img_tokens'].to(cfg.device)
            vid_tok = sample['vid_tokens'].to(cfg.device)
            
            # Forward
            img_proj = projector(img_tok.unsqueeze(0))
            vid_proj = projector(vid_tok)
            fused_img, fused_vid = fusion(img_proj, vid_proj.unsqueeze(0))
            
            for i, box in enumerate(sample['bboxes']):
                roi_feat = roi_pool_tokens_mean(
                    sample['hw'], fused_img.squeeze(0), box, sample['image_size']
                )
                roi_batch = roi_feat.unsqueeze(0).unsqueeze(0)
                outputs = head(roi_batch, roi_batch)
                
                # Store raw logits (for top-k) and softmax predictions (for binary metrics)
                logits = outputs['cls_logits'].squeeze(1)
                pred = torch.softmax(logits, dim=-1)[:, 1]
                label = sample['is_positive'][i:i+1]
                
                all_logits.append(logits.cpu())
                all_preds.append(pred.cpu())
                all_labels.append(label)
                
                # TTC
                if 'ttc' in sample and sample['ttc'] is not None:
                    ttc_pred = outputs['ttc'].squeeze().cpu()
                    ttc_label = sample['ttc'][i]
                    if not torch.isnan(torch.tensor(ttc_label)):
                        all_ttc_preds.append(ttc_pred)
                        all_ttc_labels.append(ttc_label)
    
    # Compute metrics
    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels).long()
    logits = torch.cat(all_logits)
    
    metrics = {
        'accuracy': binary_accuracy(preds, labels),
        'mAP': binary_average_precision(preds, labels),
        'top1_accuracy': binary_accuracy(preds, labels),
        'top5_accuracy': top5_accuracy(logits, labels),
    }
    
    if all_ttc_preds:
        ttc_preds = torch.stack(all_ttc_preds)
        ttc_labels = torch.tensor(all_ttc_labels)
        metrics['ttc_mae'] = ttc_mae(ttc_preds, ttc_labels)
    
    return metrics

In [ ]:
# 7.4 Full Training Loop
print("=" * 50)
print("TRAINING")
print("=" * 50)
print(f"Backbone: {cfg.backbone.upper()}")
print(f"Epochs: {cfg.epochs}")
print(f"Train samples: {len(train_ds)}")
print(f"Val samples: {len(val_ds) if val_ds else 0}")
print(f"Batch size: {cfg.batch_size}")
print(f"Estimated batches/epoch: {len(train_loader)}")
print(f"Mixed Precision (AMP): {'✓ Enabled' if scaler is not None else '✗ Disabled'}")
print(f"Multi-task Training: {'✓ Enabled' if cfg.use_multi_task else '✗ Disabled'}")
if cfg.use_multi_task:
    print(f"  Loss weights: next={cfg.loss_w_next}, noun={cfg.loss_w_noun}, verb={cfg.loss_w_verb}, ttc={cfg.loss_w_ttc}")
print("=" * 50)

# Training history (includes multi-task losses)
history = {
    'train_loss': [], 'train_loss_next': [], 'train_loss_noun': [], 
    'train_loss_verb': [], 'train_loss_ttc': [],
    'train_acc': [], 'train_mAP': [],
    'val_acc': [], 'val_mAP': [], 'val_top5_acc': [], 'val_ttc_mae': [],
    'lr': []
}

best_metric = 0.0 if cfg.monitor_metric != 'ttc_mae' else float('inf')
patience_counter = 0

for epoch in range(cfg.epochs):
    # Train
    train_metrics = train_one_epoch(epoch)
    
    history['train_loss'].append(train_metrics['loss'])
    history['train_loss_next'].append(train_metrics.get('loss_next', 0))
    history['train_loss_noun'].append(train_metrics.get('loss_noun', 0))
    history['train_loss_verb'].append(train_metrics.get('loss_verb', 0))
    history['train_loss_ttc'].append(train_metrics.get('loss_ttc', 0))
    history['train_acc'].append(train_metrics['accuracy'])
    history['train_mAP'].append(train_metrics['mAP'])
    history['lr'].append(train_metrics['lr'])
    
    print(f"\nEpoch {epoch+1}/{cfg.epochs}:")
    print(f"  Train - Loss: {train_metrics['loss']:.4f}, Acc: {train_metrics['accuracy']:.4f}, mAP: {train_metrics['mAP']:.4f}")
    if cfg.use_multi_task:
        print(f"          [next={train_metrics.get('loss_next', 0):.3f}, noun={train_metrics.get('loss_noun', 0):.3f}, verb={train_metrics.get('loss_verb', 0):.3f}, ttc={train_metrics.get('loss_ttc', 0):.3f}]")
    
    # Validate
    if (epoch + 1) % cfg.eval_every == 0 and val_ds:
        val_metrics = validate()
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_mAP'].append(val_metrics['mAP'])
        history['val_top5_acc'].append(val_metrics['top5_accuracy'])
        history['val_ttc_mae'].append(val_metrics.get('ttc_mae', 0))
        
        print(f"  Val   - Acc: {val_metrics['accuracy']:.4f}, mAP: {val_metrics['mAP']:.4f}, Top-5: {val_metrics['top5_accuracy']:.4f}")
        if 'ttc_mae' in val_metrics:
            print(f"          TTC MAE: {val_metrics['ttc_mae']:.4f}")
        
        # Check for best model
        current_metric = val_metrics.get(cfg.monitor_metric, val_metrics['mAP'])
        is_better = (current_metric > best_metric) if cfg.monitor_metric != 'ttc_mae' else (current_metric < best_metric)
        
        if is_better:
            best_metric = current_metric
            patience_counter = 0
            
            # Save best checkpoint
            torch.save({
                'epoch': epoch,
                'projector': projector.state_dict(),
                'fusion': fusion.state_dict(),
                'head': head.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scaler': scaler.state_dict() if scaler is not None else None,
                'metrics': val_metrics,
                'config': vars(cfg),
            }, ckpt_dir / 'best.pt')
            print(f"  ✓ New best {cfg.monitor_metric}: {best_metric:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= cfg.early_stopping_patience:
                print(f"\n⚠️  Early stopping at epoch {epoch+1}")
                break
    
    # Save epoch checkpoint
    torch.save({
        'epoch': epoch,
        'projector': projector.state_dict(),
        'fusion': fusion.state_dict(),
        'head': head.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict() if scaler is not None else None,
    }, ckpt_dir / f'epoch_{epoch+1:03d}.pt')

print("\n" + "=" * 50)
print("TRAINING COMPLETE")
print("=" * 50)
print(f"Backbone: {cfg.backbone.upper()}")
print(f"Best {cfg.monitor_metric}: {best_metric:.4f}")
if scaler is not None:
    print("AMP: ✓ Used mixed precision training")

# ═══════════════════════════════════════════════════════════════
# Save results for backbone comparison
# ═══════════════════════════════════════════════════════════════
final_results = {
    'backbone': cfg.backbone,
    'run_dir': str(run_dir),
    'epochs_completed': len(history['train_loss']),
    'use_amp': cfg.use_amp and scaler is not None,
    'use_multi_task': cfg.use_multi_task,
    'train_acc': history['train_acc'][-1] if history['train_acc'] else 0,
    'train_mAP': history['train_mAP'][-1] if history['train_mAP'] else 0,
    'val_acc': history['val_acc'][-1] if history['val_acc'] else 0,
    'val_mAP': history['val_mAP'][-1] if history['val_mAP'] else 0,
    'val_top5_acc': history['val_top5_acc'][-1] if history['val_top5_acc'] else 0,
    'val_ttc_mae': history['val_ttc_mae'][-1] if history['val_ttc_mae'] else 0,
    'best_metric': best_metric,
    'timestamp': datetime.now().isoformat(),
}

# Load existing comparison file or create new
if RESULTS_FILE.exists():
    with open(RESULTS_FILE, 'r') as f:
        all_results = json.load(f)
else:
    all_results = {'runs': []}

all_results['runs'].append(final_results)

with open(RESULTS_FILE, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n✓ Results saved to {RESULTS_FILE}")

TRAINING
Epochs: 2
Train samples: 1490
Val samples: 755
Batch size: 2
Estimated batches/epoch: 373


Epoch 1/2:   3%|▎         | 12/373 [00:55<28:04,  4.67s/it, loss=0.7296, lr=2.00e-03]



KeyboardInterrupt: 

---
## Part 8: Training Visualization

In [ ]:
# 8.1 Plot Training Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Loss
ax = axes[0, 0]
ax.plot(history['train_loss'], 'b-', label='Train Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy
ax = axes[0, 1]
ax.plot(history['train_acc'], 'b-', label='Train')
if history['val_acc']:
    ax.plot(range(0, len(history['val_acc']) * cfg.eval_every, cfg.eval_every), 
            history['val_acc'], 'r-', label='Val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# mAP
ax = axes[1, 0]
ax.plot(history['train_mAP'], 'b-', label='Train')
if history['val_mAP']:
    ax.plot(range(0, len(history['val_mAP']) * cfg.eval_every, cfg.eval_every),
            history['val_mAP'], 'r-', label='Val')
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.set_title('Mean Average Precision')
ax.legend()
ax.grid(True, alpha=0.3)

# Learning Rate
ax = axes[1, 1]
ax.plot(history['lr'], 'g-')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(run_dir / 'training_curves.png', dpi=150)
plt.show()

print(f"✓ Saved training curves to {run_dir / 'training_curves.png'}")

---
## Part 9: Evaluation & Inference

In [ ]:
# 9.1 Load Best Checkpoint
best_ckpt = ckpt_dir / 'best.pt'
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location=cfg.device)
    projector.load_state_dict(ckpt['projector'])
    fusion.load_state_dict(ckpt['fusion'])
    head.load_state_dict(ckpt['head'])
    print(f"✓ Loaded best checkpoint from epoch {ckpt['epoch'] + 1}")
    print(f"  Metrics: {ckpt['metrics']}")
else:
    print("⚠️  No best checkpoint found, using last epoch weights")

In [ ]:
# 9.2 Final Evaluation
print("=" * 50)
print("FINAL EVALUATION")
print("=" * 50)

if val_ds:
    final_metrics = validate()
    
    print(f"\nValidation Results:")
    print(f"  Accuracy: {final_metrics['accuracy']:.4f}")
    print(f"  mAP: {final_metrics['mAP']:.4f}")
    if 'ttc_mae' in final_metrics:
        print(f"  TTC MAE: {final_metrics['ttc_mae']:.4f}s")
    
    # Save metrics
    metrics_path = run_dir / 'final_metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(final_metrics, f, indent=2)
    print(f"\n✓ Saved metrics to {metrics_path}")

In [ ]:
# 9.3 Visualize Predictions
from PIL import Image, ImageDraw

@torch.no_grad()
def visualize_predictions(n_samples=3):
    """Visualize predictions on sample images"""
    projector.eval()
    fusion.eval()
    head.eval()
    
    fig, axes = plt.subplots(1, n_samples, figsize=(5*n_samples, 5))
    if n_samples == 1:
        axes = [axes]
    
    for idx, ax in enumerate(axes):
        sample = val_ds[idx]
        if not sample.get('valid'):
            continue
        
        img_tok = sample['img_tokens'].to(cfg.device)
        vid_tok = sample['vid_tokens'].to(cfg.device)
        
        # Forward
        img_proj = projector(img_tok.unsqueeze(0))
        vid_proj = projector(vid_tok)
        fused_img, fused_vid = fusion(img_proj, vid_proj.unsqueeze(0))
        
        # Get predictions for all candidates
        predictions = []
        for i, box in enumerate(sample['bboxes']):
            roi_feat = roi_pool_tokens_mean(
                sample['hw'], fused_img.squeeze(0), box, sample['image_size']
            )
            roi_batch = roi_feat.unsqueeze(0).unsqueeze(0)
            outputs = head(roi_batch, roi_batch)
            
            prob = torch.softmax(outputs['cls_logits'].squeeze(), dim=-1)[1].item()
            predictions.append((box, prob, sample['is_positive'][i].item()))
        
        # Load and draw on image
        img_path = sample['frame_path']
        img = Image.open(img_path).convert('RGB')
        draw = ImageDraw.Draw(img)
        
        for box, prob, gt in predictions:
            x1, y1, x2, y2 = [int(c) for c in box]
            color = 'green' if gt else 'red'
            draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
            draw.text((x1, y1-15), f"{prob:.2f}", fill=color)
        
        ax.imshow(img)
        ax.set_title(f"Sample {idx+1}: {sample['uid'][:8]}...")
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(run_dir / 'sample_predictions.png', dpi=150)
    plt.show()

if val_ds:
    visualize_predictions(3)
    print(f"✓ Saved visualizations to {run_dir / 'sample_predictions.png'}")

---
## Part 9.5: Advanced Evaluation (Hotspot Priors + CLIP Re-ranking)

The Track B evaluation pipeline supports two advanced scoring techniques from the STA literature:

1. **Hotspot Priors (PEAR-style)**: Blend model scores with empirical P(next-active | noun, verb) from training data
2. **CLIP Re-ranking**: Re-rank candidates using CLIP image-text similarity for predicted nouns

These are optional and typically used during full evaluation (`trackB_eval.py`), not notebook training.

In [ ]:
# 9.5.1 Hotspot Priors Configuration
"""
Hotspot Priors (PEAR-style): Blend model scores with prior probabilities
based on (noun, verb) pairs observed in training data.

This improves ranking by leveraging action co-occurrence statistics.
For example, if "knife" + "cut" often co-occurs with next-active candidates,
that prior knowledge can boost the score.

Usage:
    1. Build priors: python -m trackB.build_hotspot_priors
    2. Evaluate with priors: python -m trackB.trackB_eval --use_hotspot_priors
"""

# Hotspot prior settings (from trackB_eval.py)
HOTSPOT_CONFIG = {
    'enabled': False,  # Set True to use hotspot priors
    'alpha': 0.3,      # Blend weight: (1-alpha)*model + alpha*hotspot
    'prior_path': LOCAL_EXT / 'v2' / 'hotspot_priors_train_logodds_min10.json',
}

# CLIP re-ranking settings
CLIP_CONFIG = {
    'enabled': False,  # Set True to use CLIP re-ranking
    'weight': 0.3,     # Blend weight: (1-weight)*model + weight*clip
    'model': 'ViT-B/32',
}

print("=" * 60)
print("ADVANCED EVALUATION OPTIONS")
print("=" * 60)
print(f"""
Hotspot Priors (PEAR-style):
  Enabled: {HOTSPOT_CONFIG['enabled']}
  Alpha: {HOTSPOT_CONFIG['alpha']}
  Prior path: {HOTSPOT_CONFIG['prior_path']}
  
CLIP Re-ranking:
  Enabled: {CLIP_CONFIG['enabled']}
  Weight: {CLIP_CONFIG['weight']}
  Model: {CLIP_CONFIG['model']}

📝 Note: For full evaluation with these features, use:
   python -m trackB.trackB_eval --checkpoint {ckpt_dir / 'best.pt'}
""")

In [ ]:
# 9.5.2 STA Metrics: N / N+V / N+δ / All (Top-1 and Top-5)
"""
Short-Term Anticipation (STA) metrics from Ego4D challenge:

Top-1 Metrics:
- N (Noun):      Top-1 candidate has Box IoU ≥ 0.5 AND correct noun prediction
- N+V (Noun+Verb): Top-1 has IoU ≥ 0.5 AND correct noun AND verb
- N+δ (Noun+TTC): Top-1 has IoU ≥ 0.5 AND correct noun AND TTC bin
- All (N+V+δ):   Top-1 has IoU ≥ 0.5 AND correct noun, verb, AND TTC bin

Top-5 Metrics (more lenient):
- Same as above, but checks if ANY of the top-5 candidates satisfies the condition

These are frame-level metrics that require multi-task head outputs (noun/verb/TTC).
"""

def compute_sta_metrics(val_ds, val_loader, projector, fusion, head, device, iou_thresh=0.5):
    """
    Compute STA challenge metrics (N, N+V, N+δ, All) for both Top-1 and Top-5.
    Requires multi-task head with noun/verb predictions.
    
    Returns:
        dict with Top-1 and Top-5 metrics for N, NV, Nd, All
    """
    from trackB.trackB_dataset import ttc_to_bin
    
    projector.eval()
    fusion.eval()
    head.eval()
    
    # Counters for Top-1
    n_total_1 = {'N': 0, 'NV': 0, 'Nd': 0, 'All': 0}
    n_correct_1 = {'N': 0, 'NV': 0, 'Nd': 0, 'All': 0}
    
    # Counters for Top-5
    n_total_5 = {'N': 0, 'NV': 0, 'Nd': 0, 'All': 0}
    n_correct_5 = {'N': 0, 'NV': 0, 'Nd': 0, 'All': 0}
    
    def box_iou(a, b):
        """Compute IoU between two boxes (x1,y1,x2,y2)."""
        ax1, ay1, ax2, ay2 = a
        bx1, by1, bx2, by2 = b
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        iw, ih = max(0, ix2-ix1), max(0, iy2-iy1)
        inter = iw * ih
        area_a = max(0, ax2-ax1) * max(0, ay2-ay1)
        area_b = max(0, bx2-bx1) * max(0, by2-by1)
        union = area_a + area_b - inter
        return inter / union if union > 0 else 0
    
    def check_candidate(outputs, gt_noun_val, gt_verb_val, gt_ttc_bin, has_noun, has_verb):
        """Check if a candidate prediction matches GT for each metric type."""
        results = {'N': False, 'NV': False, 'Nd': False, 'All': False}
        
        if not has_noun:
            return results
        
        pred_noun = torch.argmax(outputs['noun_logits'].squeeze()).item()
        noun_match = (pred_noun == gt_noun_val)
        
        # N metric
        results['N'] = noun_match
        
        # N+V metric
        if has_verb and gt_verb_val >= 0:
            pred_verb = torch.argmax(outputs['verb_logits'].squeeze()).item()
            results['NV'] = noun_match and (pred_verb == gt_verb_val)
        
        # N+δ metric (noun + TTC bin)
        if gt_ttc_bin is not None:
            pred_ttc = outputs['ttc'].squeeze().item()
            pred_bin = ttc_to_bin(pred_ttc)
            results['Nd'] = noun_match and (pred_bin == gt_ttc_bin)
            
            # All metric (noun + verb + TTC)
            if has_verb and gt_verb_val >= 0:
                pred_verb = torch.argmax(outputs['verb_logits'].squeeze()).item()
                results['All'] = noun_match and (pred_verb == gt_verb_val) and (pred_bin == gt_ttc_bin)
        
        return results
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Computing STA metrics (Top-1 & Top-5)"):
            if not batch.get('valid'):
                continue
            
            for sample in batch['samples']:
                if not sample.get('valid'):
                    continue
                
                img_tok = sample['img_tokens'].to(device)
                vid_tok = sample['vid_tokens'].to(device)
                
                # Forward pass for fusion
                img_proj = projector(img_tok.unsqueeze(0))
                vid_proj = projector(vid_tok)
                fused_img, fused_vid = fusion(img_proj, vid_proj.unsqueeze(0))
                
                # Get predictions for all candidates with scores
                candidate_data = []
                for i, box in enumerate(sample['bboxes']):
                    roi_feat = roi_pool_tokens_mean(
                        sample['hw'], fused_img.squeeze(0), box, sample['image_size']
                    )
                    roi_batch = roi_feat.unsqueeze(0).unsqueeze(0)
                    outputs = head(roi_batch, roi_batch)
                    
                    score = torch.softmax(outputs['cls_logits'].squeeze(), dim=-1)[1].item()
                    candidate_data.append({
                        'idx': i,
                        'score': score,
                        'box': box,
                        'outputs': {k: v.clone() for k, v in outputs.items()}
                    })
                
                if not candidate_data:
                    continue
                
                # Sort by score descending
                candidate_data.sort(key=lambda x: x['score'], reverse=True)
                
                # Find GT positive candidate
                is_pos = sample.get('is_positive', sample['labels'])
                pos_indices = torch.nonzero(is_pos == 1, as_tuple=True)[0]
                if pos_indices.numel() == 0:
                    continue
                
                gt_idx = int(pos_indices[0].item())
                gt_box = tuple(sample['bboxes'][gt_idx])
                
                # Get GT labels
                gt_noun_ids = sample.get('gt_noun_id')
                gt_verb_ids = sample.get('gt_verb_id')
                gt_ttc = sample.get('ttc')
                
                gt_noun_val = int(gt_noun_ids[gt_idx].item()) if gt_noun_ids is not None else -1
                gt_verb_val = int(gt_verb_ids[gt_idx].item()) if gt_verb_ids is not None else -1
                gt_ttc_bin = ttc_to_bin(float(gt_ttc[gt_idx].item())) if gt_ttc is not None else None
                
                # Check for multi-task outputs
                has_noun = 'noun_logits' in candidate_data[0]['outputs']
                has_verb = 'verb_logits' in candidate_data[0]['outputs']
                
                if not has_noun or gt_noun_val < 0:
                    continue
                
                # ═══════════════════════════════════════════════════
                # TOP-1 EVALUATION
                # ═══════════════════════════════════════════════════
                top1 = candidate_data[0]
                iou_1 = box_iou(gt_box, top1['box'])
                
                if iou_1 >= iou_thresh:
                    results_1 = check_candidate(
                        top1['outputs'], gt_noun_val, gt_verb_val, gt_ttc_bin, has_noun, has_verb
                    )
                    
                    for key in ['N', 'NV', 'Nd', 'All']:
                        if key == 'N' or (key == 'NV' and has_verb) or (key == 'Nd' and gt_ttc_bin is not None) or (key == 'All' and has_verb and gt_ttc_bin is not None):
                            n_total_1[key] += 1
                            if results_1[key]:
                                n_correct_1[key] += 1
                
                # ═══════════════════════════════════════════════════
                # TOP-5 EVALUATION
                # ═══════════════════════════════════════════════════
                top5 = candidate_data[:min(5, len(candidate_data))]
                
                # Check if ANY of top-5 satisfies each metric
                hit_5 = {'N': False, 'NV': False, 'Nd': False, 'All': False}
                
                for cand in top5:
                    iou_k = box_iou(gt_box, cand['box'])
                    if iou_k >= iou_thresh:
                        results_k = check_candidate(
                            cand['outputs'], gt_noun_val, gt_verb_val, gt_ttc_bin, has_noun, has_verb
                        )
                        for key in ['N', 'NV', 'Nd', 'All']:
                            if results_k[key]:
                                hit_5[key] = True
                
                for key in ['N', 'NV', 'Nd', 'All']:
                    if key == 'N' or (key == 'NV' and has_verb) or (key == 'Nd' and gt_ttc_bin is not None) or (key == 'All' and has_verb and gt_ttc_bin is not None):
                        n_total_5[key] += 1
                        if hit_5[key]:
                            n_correct_5[key] += 1
    
    # Compute accuracies
    metrics = {}
    
    # Top-1 metrics
    for key in ['N', 'NV', 'Nd', 'All']:
        if n_total_1[key] > 0:
            metrics[f'{key}_top1_acc'] = n_correct_1[key] / n_total_1[key]
        else:
            metrics[f'{key}_top1_acc'] = 0.0
    
    # Top-5 metrics
    for key in ['N', 'NV', 'Nd', 'All']:
        if n_total_5[key] > 0:
            metrics[f'{key}_top5_acc'] = n_correct_5[key] / n_total_5[key]
        else:
            metrics[f'{key}_top5_acc'] = 0.0
    
    # Add counts for debugging
    metrics['counts'] = {
        'top1': {k: {'total': n_total_1[k], 'correct': n_correct_1[k]} for k in ['N', 'NV', 'Nd', 'All']},
        'top5': {k: {'total': n_total_5[k], 'correct': n_correct_5[k]} for k in ['N', 'NV', 'Nd', 'All']},
    }
    
    return metrics


def print_sta_metrics(metrics):
    """Pretty print STA metrics table."""
    print("\n" + "=" * 70)
    print("STA CHALLENGE METRICS (Ego4D Format)")
    print("=" * 70)
    
    print("\n┌" + "─" * 68 + "┐")
    print("│" + " METRIC ".center(20) + "│" + " TOP-1 ".center(22) + "│" + " TOP-5 ".center(22) + "│")
    print("├" + "─" * 20 + "┼" + "─" * 22 + "┼" + "─" * 22 + "┤")
    
    for key, name in [('N', 'Noun (N)'), ('NV', 'Noun+Verb (N+V)'), ('Nd', 'Noun+TTC (N+δ)'), ('All', 'All (N+V+δ)')]:
        top1 = metrics.get(f'{key}_top1_acc', 0) * 100
        top5 = metrics.get(f'{key}_top5_acc', 0) * 100
        print(f"│ {name:<18} │ {top1:>18.2f}% │ {top5:>18.2f}% │")
    
    print("└" + "─" * 20 + "┴" + "─" * 22 + "┴" + "─" * 22 + "┘")
    
    # Show improvement from Top-1 to Top-5
    print("\n📈 Top-5 vs Top-1 Improvement:")
    for key in ['N', 'NV', 'Nd', 'All']:
        t1 = metrics.get(f'{key}_top1_acc', 0) * 100
        t5 = metrics.get(f'{key}_top5_acc', 0) * 100
        diff = t5 - t1
        if diff > 0:
            print(f"   {key}: +{diff:.2f}%")


# Note: Only run if you have multi-task labels
print("📝 STA Metrics (N/N+V/N+δ/All) with Top-1 and Top-5 variants.")
print("   Requires multi-task heads with noun/verb predictions.")
print("\n   To compute (after training with multi-task heads):")
print("   >>> sta_metrics = compute_sta_metrics(val_ds, val_loader, projector, fusion, head, cfg.device)")
print("   >>> print_sta_metrics(sta_metrics)")

---
## Part 10: Summary & Next Steps

In [ ]:
# 10.1 Training Summary
print("=" * 60)
print("TRACK B TRAINING COMPLETE")
print("=" * 60)

print(f"""
📊 Configuration:
   Backbone: {cfg.backbone} ({spec['dim']}d)
   Fusion: {cfg.fusion_layers} layers, {cfg.fusion_heads} heads (TrackBFusion)
   Head: {cfg.num_classes} classes (TrackBHead)
   Epochs: {len(history['train_loss'])}/{cfg.epochs}
   Batch size: {cfg.batch_size}
   
📈 Final Metrics:
   Train Accuracy: {history['train_acc'][-1]:.4f}
   Train mAP: {history['train_mAP'][-1]:.4f}
""")

if history['val_mAP']:
    print(f"""   Val Accuracy:  {history['val_acc'][-1]:.4f}
   Val mAP:       {history['val_mAP'][-1]:.4f}
   Val Top-5 Acc: {history['val_top5_acc'][-1]:.4f}
   Val TTC MAE:   {history['val_ttc_mae'][-1]:.4f}
   
   Best {cfg.monitor_metric}: {best_metric:.4f}
""")

print(f"""📁 Outputs:
   Run directory: {run_dir}
   Checkpoints: {ckpt_dir}
   Best model: {ckpt_dir / 'best.pt'}

📝 TrackB Modules Used (from local_extraction/trackB):
   ✓ trackB_tokenizer.py   → TokenizerConfig, build_backbone, roi_pool_tokens_mean
   ✓ trackB_dataset.py     → TrackBDataset, trackB_collate
   ✓ trackB_fusion.py      → FusionConfig, TrackBFusion (FGTP + Cross-Attention)
   ✓ trackB_head.py        → HeadConfig, TrackBHead (Cls + TTC regression)
   ✓ trackB_metrics.py     → binary_accuracy, binary_average_precision, ttc_mae
   
🔧 Additional TrackB Tools (not used in notebook, available via CLI):
   • trackB_train_loader.py  → Full training script with all options
   • trackB_eval.py          → Full evaluation with Hotspot + CLIP re-ranking
   • build_hotspot_priors.py → Build PEAR-style priors from training data
   • extract_resnet_tokens.py → Pre-extract tokens for 120× faster training
   • summarize_ablation.py   → Compare VideoMAE vs ResNet18 runs
   • trackB_plots.py         → Generate training curve plots
   • videomae/               → VideoMAE backbone implementation
""")

print("""
🚀 Next Steps:
   1. Compare backbones:
      - Set BACKBONE = 'videomae' and re-run
      - Run Part 11 comparison cells
      
   2. Full evaluation with advanced scoring:
      python -m trackB.trackB_eval --checkpoint <path> \\
          --use_hotspot_priors --use_clip_rerank
      
   3. Pre-extract tokens for faster training:
      python -m trackB.extract_resnet_tokens
      
   4. Generate ablation summary:
      python -m trackB.summarize_ablation
""")

---
## Part 11: Backbone Comparison (ResNet18 vs VideoMAE)

In [ ]:
# 11.1 Load and Compare Results from Both Backbones
"""
This cell compares results between ResNet18 and VideoMAE backbone runs.
Run this after training with BOTH backbones to see the comparison.
"""
import json
import pandas as pd

print("=" * 70)
print("BACKBONE COMPARISON: ResNet18 vs VideoMAE")
print("=" * 70)

# Load comparison results
RESULTS_FILE = PATHS['trackB_runs'] / "backbone_comparison.json"

if not RESULTS_FILE.exists():
    print("⚠️  No comparison file found yet.")
    print(f"   File: {RESULTS_FILE}")
    print("\n   Run training with BOTH backbones to generate comparison data:")
    print("   1. Set BACKBONE = 'resnet18', run all cells")
    print("   2. Set BACKBONE = 'videomae', run all cells")
    print("   3. Run this comparison cell")
else:
    with open(RESULTS_FILE, 'r') as f:
        all_results = json.load(f)
    
    runs = all_results['runs']
    print(f"\n📊 Found {len(runs)} training run(s):\n")
    
    # Get latest run for each backbone
    resnet_runs = [r for r in runs if r['backbone'] == 'resnet18']
    videomae_runs = [r for r in runs if r['backbone'] == 'videomae']
    
    # Build comparison table
    comparison_data = []
    
    for backbone_name, backbone_runs in [('resnet18', resnet_runs), ('videomae', videomae_runs)]:
        if backbone_runs:
            latest = backbone_runs[-1]  # Most recent run
            comparison_data.append({
                'Backbone': backbone_name.upper(),
                'Train Acc': f"{latest['train_acc']:.4f}",
                'Train mAP': f"{latest['train_mAP']:.4f}",
                'Val Acc': f"{latest['val_acc']:.4f}",
                'Val mAP': f"{latest['val_mAP']:.4f}",
                'Val Top-5': f"{latest['val_top5_acc']:.4f}",
                'TTC MAE': f"{latest['val_ttc_mae']:.4f}",
                'Epochs': latest['epochs_completed'],
            })
    
    if len(comparison_data) >= 2:
        print("┌" + "─" * 68 + "┐")
        print("│" + " COMPARISON TABLE ".center(68) + "│")
        print("├" + "─" * 68 + "┤")
        
        # Headers
        headers = ['Backbone', 'Val Acc', 'Val mAP', 'Val Top-5', 'TTC MAE']
        header_row = "│ " + " │ ".join([h.center(12) for h in headers]) + " │"
        print(header_row)
        print("├" + "─" * 68 + "┤")
        
        # Data rows
        for row in comparison_data:
            data_row = "│ " + " │ ".join([
                row['Backbone'].center(12),
                row['Val Acc'].center(12),
                row['Val mAP'].center(12),
                row['Val Top-5'].center(12),
                row['TTC MAE'].center(12),
            ]) + " │"
            print(data_row)
        
        print("└" + "─" * 68 + "┘")
        
        # Compute differences
        r = next((d for d in comparison_data if d['Backbone'] == 'RESNET18'), None)
        v = next((d for d in comparison_data if d['Backbone'] == 'VIDEOMAE'), None)
        
        if r and v:
            print("\n📈 Improvement (VideoMAE vs ResNet18):")
            
            r_map = float(r['Val mAP'])
            v_map = float(v['Val mAP'])
            r_top5 = float(r['Val Top-5'])
            v_top5 = float(v['Val Top-5'])
            
            map_diff = v_map - r_map
            top5_diff = v_top5 - r_top5
            
            map_sign = "+" if map_diff >= 0 else ""
            top5_sign = "+" if top5_diff >= 0 else ""
            
            print(f"   Val mAP:   {map_sign}{map_diff:.4f} ({map_sign}{map_diff*100:.2f}%)")
            print(f"   Val Top-5: {top5_sign}{top5_diff:.4f} ({top5_sign}{top5_diff*100:.2f}%)")
            
            winner = "VIDEOMAE" if v_map > r_map else "RESNET18" if r_map > v_map else "TIE"
            print(f"\n   🏆 Better mAP: {winner}")
    
    elif len(comparison_data) == 1:
        print(f"   Only {comparison_data[0]['Backbone']} results found.")
        print("   Run the other backbone to enable comparison.")
        
        print(f"\n   Current results ({comparison_data[0]['Backbone']}):")
        for k, v in comparison_data[0].items():
            print(f"     {k}: {v}")
    
    else:
        print("   No valid runs found in comparison file.")

print("\n" + "=" * 70)

In [ ]:
# 11.2 Visualize Backbone Comparison
"""
Bar chart comparison of key metrics between backbones.
"""
import matplotlib.pyplot as plt
import numpy as np

if RESULTS_FILE.exists():
    with open(RESULTS_FILE, 'r') as f:
        all_results = json.load(f)
    
    runs = all_results['runs']
    resnet_runs = [r for r in runs if r['backbone'] == 'resnet18']
    videomae_runs = [r for r in runs if r['backbone'] == 'videomae']
    
    if resnet_runs and videomae_runs:
        r = resnet_runs[-1]
        v = videomae_runs[-1]
        
        # Metrics to compare
        metrics = ['Val Accuracy', 'Val mAP', 'Val Top-5']
        resnet_vals = [r['val_acc'], r['val_mAP'], r['val_top5_acc']]
        videomae_vals = [v['val_acc'], v['val_mAP'], v['val_top5_acc']]
        
        x = np.arange(len(metrics))
        width = 0.35
        
        fig, ax = plt.subplots(figsize=(10, 6))
        bars1 = ax.bar(x - width/2, resnet_vals, width, label='ResNet18', color='#2196F3')
        bars2 = ax.bar(x + width/2, videomae_vals, width, label='VideoMAE', color='#4CAF50')
        
        ax.set_ylabel('Score')
        ax.set_title('Track B: ResNet18 vs VideoMAE Backbone Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(metrics)
        ax.legend()
        ax.set_ylim(0, 1.0)
        ax.grid(axis='y', alpha=0.3)
        
        # Add value labels on bars
        def add_labels(bars):
            for bar in bars:
                height = bar.get_height()
                ax.annotate(f'{height:.3f}',
                           xy=(bar.get_x() + bar.get_width() / 2, height),
                           xytext=(0, 3),
                           textcoords="offset points",
                           ha='center', va='bottom', fontsize=9)
        
        add_labels(bars1)
        add_labels(bars2)
        
        plt.tight_layout()
        
        # Save comparison figure
        comparison_fig_path = PATHS['trackB_runs'] / 'backbone_comparison.png'
        plt.savefig(comparison_fig_path, dpi=150, bbox_inches='tight')
        plt.show()
        
        print(f"✓ Comparison chart saved to {comparison_fig_path}")
    else:
        print("⚠️  Need results from BOTH backbones to create comparison chart.")
        print("   Run training with both BACKBONE = 'resnet18' and 'videomae'.")
else:
    print("⚠️  No comparison results file found.")